# Insect Trajectory Prediction — Multi-step & the KUN scheme

**TP3 (part 2) — Unified Notebook (PyTorch)**

- Binôme 1:
- Binôme 2:


`Objectives`: forecast a **full horizon** `H` of the 2-D insect trajectory in one go (**direct
multi-output**), and add the **KUN (Kernel U-Net)** scheme.

This is the sequel to [`03_insect_trajectory.ipynb`](03_insect_trajectory.ipynb): **same dataset,
same models**, but here every model emits the **whole horizon at once** instead of being rolled out.

1. Slice the series into `(lookback L -> horizon H)` samples.
2. Build several **direct multi-output** models: `Linear` (worked example), then `MLP` / `RNN` /
   `LSTM` / `GRU` / `Transformer` (**for you to implement**).
3. Build **KUN**: a U-shaped hierarchy whose per-node **kernel** is pluggable — `linear`
   (worked example), then `mlp` / `attention` (**for you to implement**). The kernel plays the
   same role the standalone architectures do, but *inside* the hierarchy.
4. Train, benchmark, and plot — exactly like notebook 3.
5. Run a **scaling experiment**: grow `L = H` from 32 to 256 and watch the error.


## Notebook structure

Run **top to bottom**. **Section 1** collects the hyper-parameters. **Section 2** simulates the
insect, **Section 3** builds standardised train / test sets, **Section 4** slices `(L -> H)`
windows. **Section 5** defines the direct models (`Linear` given, the rest `TODO`). **Section 6**
defines **KUN** (the U-Net body is given; the `linear` kernel is given, `mlp` / `attention` are
`TODO`). **Section 7** is the train / evaluate utilities, **Section 8** trains every enabled model
and reports the benchmark, **Section 9** plots the per-horizon error and forecast trajectories, and
**Section 10** runs the `L = H` scaling experiment. **Section 11** is the conclusion.

Everything runs out of the box with `Linear` + `KUN-linear`; as you implement and enable more
models in the `MODELS` factory, the tables and plots update automatically.


In [ ]:
import random
from math import sin, cos

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(0)
np.random.seed(0)
print('Using device:', device)

## 1. Parameters

All hyper-parameters in one place. `predict_len` (`H`) is now a **direct** forecast horizon.

In [ ]:
# ---- Dataset ---------------------------------------------------
max_t        = 100      # simulation horizon (time units)
delta_t      = 0.01     # time step
features_len = 2        # channels C : (x, y)

sequence_len = 32       # lookback L : length of the input window
predict_len  = 32       # horizon  H : number of steps to forecast at once

# ---- Training --------------------------------------------------
scale        = 0.2      # fraction of the simulated trajectory to use (1 = all)
train_prop   = 0.8      # train / test split ratio
batch_size   = 32
epochs       = 20
lr           = 1e-3

## 2. Insect simulation

Identical to notebook 3: a smooth, quasi-periodic 2-D motion built from sines and cosines.

In [ ]:
def insect_init(s=122):
    if s > 0:
        random.seed(s)
    insect_init.params_x = [random.gauss(0., 1.) for _ in range(8)]
    insect_init.params_y = [random.gauss(0., 1.) for _ in range(8)]


def insect_move(t):
    [ax1, ax2, ax3, ax4, kx1, kx2, kx3, kx4] = insect_init.params_x
    [ay1, ay2, ay3, ay4, ky1, ky2, ky3, ky4] = insect_init.params_y

    x = ax1*sin(t*(kx1+20)) + ax2*cos(t*(kx2+10)) + ax3*sin(t*(kx3+5)) + ax4*cos(t*(kx4+5))
    y = ay1*cos(t*(ky1+20)) + ay2*sin(t*(ky2+10)) + ay3*cos(t*(ky3+5)) + ay4*sin(t*(ky4+5))
    return x, y

## 3. Build the dataset (generate -> split -> standardise)

Generate the trajectory, split it chronologically, and standardise per channel with the **train**
statistics only (no leakage).

In [ ]:
# ---- Generate the trajectory
insect_init(s=16)
positions = [insect_move(t) for t in np.arange(0., max_t, delta_t)]

# ---- Keep a fraction, then split chronologically into train / test
n       = int(len(positions) * scale)
dataset = np.array(positions[:n], dtype='float32')

k       = int(len(dataset) * train_prop)
x_train = dataset[:k]
x_test  = dataset[k:]

# ---- Standardisation (per channel, TRAIN stats only) -----------
mean = x_train.mean(axis=0)
std  = x_train.std(axis=0)
x_train = (x_train - mean) / std
x_test  = (x_test  - mean) / std

print('Train shape :', x_train.shape)
print('Test  shape :', x_test.shape)

In [ ]:
plt.figure(figsize=(6, 5))
plt.plot(x_train[:, 0], x_train[:, 1], c='tab:blue', lw=1, alpha=0.6, label='Train')
plt.plot(x_test[:, 0],  x_test[:, 1],  c='tab:red',  lw=1, alpha=0.6, label='Test')
plt.xlabel('x (normalized)'); plt.ylabel('y (normalized)')
plt.title('Insect trajectory: train / test')
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

## 4. Multi-step windows (L -> H)

Each sample maps a lookback window to the **whole** horizon:
`X[i] = data[i : i+L]` (shape `L x C`) -> `Y[i] = data[i+L : i+L+H]` (shape `H x C`).

In [ ]:
def make_windows(data, L, H):
    """Return X (N, L, C) and Y (N, H, C): window -> next H points."""
    X, Y = [], []
    for i in range(len(data) - L - H + 1):
        X.append(data[i:i+L])
        Y.append(data[i+L:i+L+H])
    return np.array(X, dtype='float32'), np.array(Y, dtype='float32')


Xtr, Ytr = make_windows(x_train, sequence_len, predict_len)
Xte, Yte = make_windows(x_test,  sequence_len, predict_len)
print('X_train:', Xtr.shape, ' Y_train:', Ytr.shape)
print('X_test :', Xte.shape, ' Y_test :', Yte.shape)

train_loader = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(Ytr)),
                          batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(TensorDataset(torch.from_numpy(Xte), torch.from_numpy(Yte)),
                          batch_size=batch_size, shuffle=False)

## 5. Models (direct multi-output)

Every model maps an input window `(B, L, C)` to the **whole horizon** `(B, H, C)`.

`Linear` is provided as a **worked example**. Implement the other models (`MLP`, `RNN`, `LSTM`,
`GRU`, `Transformer`) yourself: replace every `# >>> YOUR CODE HERE <<<` block and delete the
`raise NotImplementedError` line. They share the same interface — `(B, L, C) -> (B, H, C)`.

In [ ]:
class DirectLinear(nn.Module):
    """Pure linear baseline (worked example): flatten the window -> the whole horizon."""
    def __init__(self, L, H, C):
        super().__init__()
        self.H, self.C = H, C
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(L * C, H * C))

    def forward(self, x):                       # x: (B, L, C)
        return self.net(x).view(-1, self.H, self.C)


# ======================================================================
# TODO (PLACEHOLDER): implement the models below.
# Same interface as DirectLinear: input (B, L, C) -> output (B, H, C).
# ======================================================================

class DirectMLP(nn.Module):
    def __init__(self, L, H, C, hidden=128):
        super().__init__()
        self.H, self.C = H, C
        # >>> YOUR CODE HERE <<< : Flatten -> Linear -> ReLU -> ... -> Linear(H*C)
        raise NotImplementedError("TODO: build the MLP")

    def forward(self, x):
        # >>> YOUR CODE HERE <<< : run net, then .view(-1, self.H, self.C)
        raise NotImplementedError


class RecurrentDirect(nn.Module):
    """RNN / LSTM / GRU encoder + linear head on the last hidden state."""
    def __init__(self, cell, L, H, C, hidden=64):
        super().__init__()
        self.H, self.C = H, C
        # >>> YOUR CODE HERE <<< : pick nn.RNN / nn.LSTM / nn.GRU from `cell` (batch_first=True),
        #                          then a Linear(hidden, H*C) head
        raise NotImplementedError("TODO: build the recurrent model")

    def forward(self, x):
        # >>> YOUR CODE HERE <<< : run rnn, take the LAST time step, apply head, reshape (B,H,C)
        raise NotImplementedError


class DirectTransformer(nn.Module):
    def __init__(self, L, H, C, d_model=64, nhead=4, layers=2):
        super().__init__()
        self.H, self.C = H, C
        # >>> YOUR CODE HERE <<< : input projection (C->d_model) + positional encoding
        #                          + TransformerEncoder + Linear head to H*C
        raise NotImplementedError("TODO: build the Transformer")

    def forward(self, x):
        # >>> YOUR CODE HERE <<<
        raise NotImplementedError

## 6. KUN (Kernel U-Net)

**KUN = a U-shaped hierarchy + a pluggable kernel at every node**, with **direct multi-output**:
embed `C -> d_model`; the **encoder** chunks `k` neighbouring units into one (down-sampling, coarser
scales); the **decoder** mirrors it, expanding back and adding the matching encoder feature
(**skip connection**); a head maps the reconstruction to the whole horizon `H*C`.

The **U-Net body is given**. The part you tune is the **kernel** `K_l` at each node — the small
function that maps one group of units to another. The `linear` kernel is the **worked example**;
implement the `mlp` and `attention` kernels yourself. This is exactly the *"treat the model as a
kernel"* idea: the kernel is where `MLP` / attention plug in, just like the standalone models above.

In [ ]:
def default_patches(L):
    """Factor L into down-sampling rates (4s, then 2 if needed). prod == L."""
    ps, x = [], L
    while x > 1:
        if   x % 4 == 0: ps.append(4); x //= 4
        elif x % 2 == 0: ps.append(2); x //= 2
        else: break
    return tuple(ps)


class Kernel(nn.Module):
    """A pluggable node operator for KUN: maps in_dim -> out_dim per unit."""
    def __init__(self, in_dim, out_dim, kind='linear', hidden=64):
        super().__init__()
        self.kind = kind
        if kind == 'linear':
            self.f = nn.Linear(in_dim, out_dim)               # worked example
        elif kind == 'mlp':
            # >>> YOUR CODE HERE <<< : Linear(in_dim, hidden) -> GELU -> Linear(hidden, out_dim)
            raise NotImplementedError("TODO: MLP kernel")
        elif kind == 'attention':
            # >>> YOUR CODE HERE <<< : a small self-attention over the units, then project to out_dim
            raise NotImplementedError("TODO: attention kernel")
        else:
            raise ValueError(f'unknown kernel: {kind}')

    def forward(self, x):
        return self.f(x)


class KernelUNet(nn.Module):
    """Hierarchical, direct multi-output forecaster (GIVEN): (B, L, C) -> (B, H, C)."""
    def __init__(self, L, H, C, d_model=32, kernel='linear', patch_sizes=None):
        super().__init__()
        ps = tuple(patch_sizes) if patch_sizes else default_patches(L)
        assert int(np.prod(ps)) == L, 'prod(patch_sizes) must equal L'
        self.L, self.H, self.C = L, H, C
        self.ps, self.d = list(ps), d_model
        self.embed = nn.Linear(C, d_model)
        self.enc = nn.ModuleList([Kernel(k * d_model, d_model, kernel) for k in ps])
        self.dec = nn.ModuleList([Kernel(d_model, k * d_model, kernel) for k in reversed(ps)])
        self.head = nn.Linear(L * d_model, H * C)             # direct multi-output

    def forward(self, x):
        B = x.size(0)
        z = self.embed(x)
        skips = [z]
        for k, ker in zip(self.ps, self.enc):                # encode: down-sample
            T = z.size(1)
            z = ker(z.reshape(B, T // k, k * self.d))
            skips.append(z)
        skips = skips[:-1]
        for k, ker in zip(reversed(self.ps), self.dec):      # decode: up-sample
            T = z.size(1)
            z = ker(z).reshape(B, T * k, self.d)
            z = z + skips.pop()                              # additive skip connection
        return self.head(z.reshape(B, -1)).view(B, self.H, self.C)


# Factory: name -> builder(L, H, C). Only Linear + KUN-linear are active by default.
# TODO (PLACEHOLDER): uncomment each line once that model / kernel is implemented.
MODELS = {
    'Linear':        lambda L, H, C: DirectLinear(L, H, C),
    # 'MLP':           lambda L, H, C: DirectMLP(L, H, C),                       # TODO
    # 'RNN':           lambda L, H, C: RecurrentDirect('RNN',  L, H, C),         # TODO
    # 'LSTM':          lambda L, H, C: RecurrentDirect('LSTM', L, H, C),         # TODO
    # 'GRU':           lambda L, H, C: RecurrentDirect('GRU',  L, H, C),         # TODO
    # 'Transformer':   lambda L, H, C: DirectTransformer(L, H, C),              # TODO
    'KUN-linear':    lambda L, H, C: KernelUNet(L, H, C, kernel='linear'),
    # 'KUN-mlp':       lambda L, H, C: KernelUNet(L, H, C, kernel='mlp'),         # TODO
    # 'KUN-attention': lambda L, H, C: KernelUNet(L, H, C, kernel='attention'),   # TODO
}

## 7. Experiment: train & evaluate

Reusable utilities (same shape as notebook 3): `evaluate` returns the multi-step `(MSE, MAE)`;
`train_model` trains with Adam + MSE and records the per-epoch train / val loss. `direct_predict`
and `per_horizon_rmse` are used later for the horizon analysis.

In [ ]:
def evaluate(model, loader):
    """Return multi-step (MSE, MAE) over a data loader."""
    model.eval()
    mse_sum, mae_sum, n = 0.0, 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)                                  # (B, H, C)
            mse_sum += nn.functional.mse_loss(pred, yb, reduction='sum').item()
            mae_sum += nn.functional.l1_loss(pred, yb, reduction='sum').item()
            n += yb.numel()
    return mse_sum / n, mae_sum / n


def train_model(model, train_loader, test_loader, epochs=epochs, lr=lr, verbose=False):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    history = {'train': [], 'val': []}
    for ep in range(epochs):
        model.train()
        run = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            run += loss.item() * len(xb)
        history['train'].append(run / len(train_loader.dataset))
        history['val'].append(evaluate(model, test_loader)[0])
        if verbose:
            print(f'  epoch {ep+1:2d}/{epochs}  val {history["val"][-1]:.4f}')
    return model, history


@torch.no_grad()
def direct_predict(model, X):
    model.eval()
    return model(torch.from_numpy(X).to(device)).cpu().numpy()


def per_horizon_rmse(pred, true):
    """(N, H, C) -> RMSE at each horizon step, shape (H,)."""
    return np.sqrt(((pred - true) ** 2).mean(axis=(0, 2)))

## 8. Compare the models

Train every model in `MODELS`, then summarise. By default only `Linear` and `KUN-linear` run; the
table and plots grow automatically as you enable more.

In [ ]:
results = {}
for name, build in MODELS.items():
    print(f'Training {name} ...')
    torch.manual_seed(0)
    model, history = train_model(build(sequence_len, predict_len, features_len),
                                 train_loader, test_loader)
    mse, mae = evaluate(model, test_loader)
    results[name] = {'model': model, 'history': history, 'mse': mse, 'mae': mae}

# ---- Benchmark table (multi-step) ----
benchmark = pd.DataFrame(
    [{'Model': n, f'{predict_len}-step MSE': r['mse'], f'{predict_len}-step MAE': r['mae']}
     for n, r in results.items()]
).set_index('Model').sort_values(f'{predict_len}-step MSE')
benchmark

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for name, r in results.items():
    ax1.plot(r['history']['val'], label=name)
ax1.set_xlabel('epoch'); ax1.set_ylabel('validation MSE')
ax1.set_title('Validation loss curves'); ax1.legend(); ax1.grid(True)

names = list(results.keys())
ax2.bar(names, [results[n]['mse'] for n in names], color='tab:blue')
ax2.set_ylabel('test MSE'); ax2.set_title(f'{predict_len}-step test MSE')
ax2.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

## 9. Multi-step forecast: per-horizon error & trajectories

The **per-horizon curve** shows how the error grows with how far ahead we predict. Then we plot a
few predicted vs. true trajectories for the best model.

In [ ]:
# ---- per-horizon RMSE + add the mean to the benchmark ----
for name, r in results.items():
    r['per_horizon'] = per_horizon_rmse(direct_predict(r['model'], Xte), Yte)
benchmark['mean RMSE'] = [results[n]['per_horizon'].mean() for n in benchmark.index]
benchmark = benchmark.sort_values('mean RMSE')
best_name = benchmark.index[0]
print('Best model:', best_name)

plt.figure(figsize=(9, 5))
steps = np.arange(1, predict_len + 1)
for name, r in sorted(results.items(), key=lambda kv: kv[1]['per_horizon'].mean()):
    style = '-o' if name.startswith('KUN') else '-'
    plt.plot(steps, r['per_horizon'], style, ms=3, label=name)
plt.xlabel('forecast step h'); plt.ylabel('RMSE')
plt.title('Error vs. forecast horizon (lower & flatter is better)')
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()
benchmark

In [ ]:
# ---- trajectories of the best model ----
best_model = results[best_name]['model']
starts = np.linspace(0, len(Xte) - 1, 3, dtype=int)

fig, axes = plt.subplots(1, len(starts), figsize=(15, 4.5))
for ax, s in zip(axes, starts):
    window = Xte[s]
    true   = Yte[s]
    pred   = direct_predict(best_model, Xte[s:s+1])[0]
    ax.plot(window[:, 0], window[:, 1], 'o-', c='tab:blue',  ms=3, label='input')
    ax.plot(true[:, 0],   true[:, 1],   'o-', c='tab:green', ms=3, label='true')
    ax.plot(pred[:, 0],   pred[:, 1],   'x--', c='tab:red',  ms=4, label='predicted')
    ax.set_title(f'start={s}'); ax.grid(True)
axes[0].legend()
fig.suptitle(f'{best_name}: direct {predict_len}-step forecast')
plt.tight_layout(); plt.show()

## 10. Scaling experiment: `L = H` from 32 to 256

How does performance change as we grow both the lookback and the horizon together? We retrain every
enabled model on the **full** trajectory (so even `L = H = 256` has enough windows) for
`L = H in {32, 64, 128, 256}` and plot the mean RMSE.

> This trains every enabled model once per `L`. Cost grows with `L`; with only `Linear` +
> `KUN-linear` enabled it runs in a few minutes on CPU. Reduce `sweep_epochs` if needed.

In [ ]:
sweep_Ls    = [32, 64, 128, 256]
sweep_epochs = 10

full = np.array(positions, dtype='float32')          # the whole simulated trajectory

def build_LH(data, L, H, train_prop=train_prop):
    """Full pipeline for a given (L, H): split, standardise (train stats), window."""
    kk = int(len(data) * train_prop)
    tr, te = data[:kk], data[kk:]
    m, s = tr.mean(0), tr.std(0)
    tr, te = (tr - m) / s, (te - m) / s
    return make_windows(tr, L, H) + make_windows(te, L, H)   # Xtr,Ytr,Xte,Yte

sweep = {name: [] for name in MODELS}
for L in sweep_Ls:
    H = L
    Xa, Ya, Xb, Yb = build_LH(full, L, H)
    tl = DataLoader(TensorDataset(torch.from_numpy(Xa), torch.from_numpy(Ya)),
                    batch_size=batch_size, shuffle=True)
    vl = DataLoader(TensorDataset(torch.from_numpy(Xb), torch.from_numpy(Yb)),
                    batch_size=batch_size, shuffle=False)
    for name, build in MODELS.items():
        torch.manual_seed(0)
        model, _ = train_model(build(L, H, features_len), tl, vl, epochs=sweep_epochs)
        rmse = per_horizon_rmse(direct_predict(model, Xb), Yb).mean()
        sweep[name].append(rmse)
        print(f'L=H={L:3d}  {name:12s}  mean RMSE {rmse:.4f}')

plt.figure(figsize=(8, 5))
for name, vals in sweep.items():
    plt.plot(sweep_Ls, vals, '-o', label=name)
plt.xlabel('L = H  (lookback = horizon)'); plt.ylabel('mean RMSE')
plt.xscale('log', base=2); plt.xticks(sweep_Ls, sweep_Ls)
plt.title('Scaling: error vs. window / horizon length')
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

## 11. Conclusion

_Fill in from your own run._

**Models implemented:** Linear (given), KUN-linear (given), ... _(list the ones you added)_

**Benchmark summary** (Section 8-9):
- Best model overall (lowest mean RMSE): ...
- How do the standard architectures rank against KUN? ...

**Observations:**
- Does KUN's per-horizon curve stay flatter than the others as `h` grows? ...
- KUN: did swapping the `linear` kernel for `mlp` / `attention` help? Was it worth the cost? ...

**Scaling experiment (Section 10):**
- How does the error change as `L = H` grows from 32 to 256? ...
- Does KUN keep its edge at longer windows / horizons? ...

**Takeaway:** **direct multi-output** forecasts the whole horizon at once, and **KUN** adds a
multi-scale hierarchy with a pluggable kernel — see [the real KUN](https://jiangyou2025.github.io/kun/zh/kun/).
